# CLIK Workshop 2 - End-to-End ML di Snowflake (Enhanced)## Use case: Probability of Default (PD) - Credit Default Prediction**Enhancements over v1:**- **Snowpark ML** preprocessing (distributed, no pandas bottleneck)- **Feature Store** (entity, feature view, dataset generation)- **ML Monitoring / MLOps** (model monitor, drift detection)**Alur:**1. Setup & Session2. Feature Store: Entity + Feature View3. Dataset generation dari Feature Store4. Preprocessing dengan Snowpark ML5. Model training: XGBoost & LightGBM (Snowpark ML distributed)6. Logistic Regression (stepwise - pandas, karena stepwise belum ada di Snowpark ML)7. Evaluasi & perbandingan8. Register ke Model Registry9. ML Monitoring (Model Monitor)10. Test inferenceRef:- https://www.snowflake.com/en/developers/guides/end-to-end-ml-workflow/- https://www.snowflake.com/en/developers/guides/intro-to-online-feature-store-in-snowflake/

## 1. Setup & Session

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom snowflake.snowpark.context import get_active_sessionfrom snowflake.snowpark.functions import col, avg, when, litsession = get_active_session()session.use_database('CLIK_WORKSHOP2')session.use_schema('PUBLIC')session.use_warehouse('GEN2_SMALL')print('Connected:', session.get_current_database(), session.get_current_schema())DB = 'CLIK_WORKSHOP2'SCHEMA = 'PUBLIC'WH = 'GEN2_SMALL'TARGET = 'DEFAULT_FLAG'

## 2. Feature Store: Entity & Feature ViewMenggunakan `snowflake.ml.feature_store` untuk:- Mendaftarkan **Entity** (subject_id sebagai primary key)- Membuat **Feature View** dari engineered features- Track lineage & metadata fitur secara terpusat

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationModefs = FeatureStore(    session=session,    database=DB,    name=SCHEMA,    default_warehouse=WH,    creation_mode=CreationMode.CREATE_IF_NOT_EXIST)print('Feature Store initialized')

In [ ]:
# Register Entity - SUBJECT_ID sebagai join keytry:    subject_entity = fs.get_entity('CREDIT_SUBJECT')    print('Retrieved existing entity')except:    subject_entity = Entity(        name='CREDIT_SUBJECT',        join_keys=['SUBJECT_ID'],        desc='Credit bureau subject - satu baris per debitur'    )    fs.register_entity(subject_entity)    print('Registered new entity: CREDIT_SUBJECT')

In [ ]:
# Buat Feature View dari SUBJECT_FEATURES# Pilih fitur yang akan dikelola di Feature Storesnow_df = session.table('SUBJECT_FEATURES')CORE_FEATURES = [    'SUBJECT_ID', 'AGE', 'MONTHLY_INCOME', 'NUM_ACTIVE_LOANS',    'NUM_CREDIT_CARDS', 'NUM_FINTECH_LOANS', 'NUM_BNPL_ACCOUNTS',    'NUM_LENDERS', 'NUM_INQUIRIES_3M', 'NUM_INQUIRIES_12M',    'MAX_DPD_12M', 'MAX_DPD_24M', 'NUM_LATE_PMT_12M',    'OLDEST_ACCT_AGE_MONTHS', 'AVG_ACCT_AGE_MONTHS',    'TOTAL_OUTSTANDING', 'TOTAL_CREDIT_LIMIT', 'CREDIT_UTILIZATION',    'MONTHLY_INSTALLMENT', 'KOL_STATUS', 'DEPENDENTS',    'GENDER', 'EMPLOYMENT_TYPE', 'EDUCATION', 'REGION_CODE',    'CC_OUTSTANDING', 'KTA_OUTSTANDING', 'KPR_OUTSTANDING',    'CC_UTIL', 'KTA_UTIL', 'PAYMENT_RATIO_3M', 'PAYMENT_RATIO_12M',    'ONTIME_RATIO_12M', 'REVOLVING_RATIO', 'DTI_RATIO']# DTI_RATIO belum ada di tabel, kita buat sebagai derived featurefeature_df = snow_df.select(CORE_FEATURES[:-1]).with_column(    'DTI_RATIO',    col('MONTHLY_INSTALLMENT') / when(col('MONTHLY_INCOME') == 0, lit(1)).otherwise(col('MONTHLY_INCOME')))print('Feature DataFrame columns:', len(feature_df.columns))feature_df.show(3)

In [ ]:
# Register Feature Viewcredit_fv = FeatureView(    name='CREDIT_BUREAU_FEATURES',    entities=[subject_entity],    feature_df=feature_df,    refresh_freq='1 day',    desc='Core credit bureau features for PD modeling (35 features)')# Attach feature descriptionscredit_fv = credit_fv.attach_feature_desc({    'AGE': 'Usia debitur',    'MONTHLY_INCOME': 'Pendapatan bulanan (IDR)',    'CREDIT_UTILIZATION': 'Rasio penggunaan kredit',    'MAX_DPD_12M': 'Hari keterlambatan maksimum 12 bulan terakhir',    'KOL_STATUS': 'Kolektibilitas OJK (1=lancar, 5=macet)',    'DTI_RATIO': 'Debt-to-income ratio',    'NUM_INQUIRIES_12M': 'Jumlah inquiry biro 12 bulan terakhir',})registered_fv = fs.register_feature_view(credit_fv, version='1', overwrite=True)print('Feature View registered:', registered_fv.name, registered_fv.version)

## 3. Generate Training Dataset dari Feature StoreGunakan `fs.generate_dataset()` — menggabungkan spine (label) dengan feature view secara otomatis.

In [ ]:
# Spine = ID + label (tanpa fitur, fitur ditarik dari Feature Store)spine_df = snow_df.select('SUBJECT_ID', 'DEFAULT_FLAG').sample(n=300000)print('Spine rows:', spine_df.count())# Generate dataset - Feature Store auto-join fitur ke spineds = fs.generate_dataset(    name='CLIK_PD_TRAINING_SET',    spine_df=spine_df,    features=[registered_fv],    spine_label_cols=['DEFAULT_FLAG'],    desc='PD training dataset from Feature Store')# Read sebagai Snowpark DataFrameds_sp = ds.read.to_snowpark_dataframe()print('Training dataset shape:', ds_sp.count(), 'rows x', len(ds_sp.columns), 'cols')ds_sp.show(3)

## 4. Preprocessing dengan Snowpark MLGanti `sklearn.preprocessing` → `snowflake.ml.modeling.preprocessing`.Process berjalan **distributed di Snowflake** (bukan di client/pandas).

In [ ]:
import snowflake.ml.modeling.preprocessing as snowmlfrom snowflake.snowpark.types import StringType# Identifikasi kolom kategorikalCAT_COLS = [c.name for c in ds_sp.schema if c.datatype == StringType() and c.name != 'SUBJECT_ID']NUM_COLS = [c for c in ds_sp.columns if c not in CAT_COLS + ['SUBJECT_ID', 'DEFAULT_FLAG']]print(f'Categorical: {len(CAT_COLS)} -> {CAT_COLS}')print(f'Numeric: {len(NUM_COLS)}')

In [ ]:
# OneHotEncoder (Snowpark ML) - berjalan distributed di warehouseohe = snowml.OneHotEncoder(    input_cols=CAT_COLS,    output_cols=CAT_COLS,    drop_input_cols=True)ds_encoded = ohe.fit(ds_sp).transform(ds_sp)print('After OHE:', len(ds_encoded.columns), 'columns')# StandardScaler (Snowpark ML) - opsional untuk tree models, wajib untuk LR# Untuk XGB/LGBM kita skip scaling, untuk LR stepwise tetap pakai# (karena stepwise LR butuh normalisasi)

In [ ]:
# Train/Test split menggunakan random_split (native Snowpark, no pandas!)train_sp, test_sp = ds_encoded.random_split(weights=[0.75, 0.25], seed=42)train_sp = train_sp.fillna(0)test_sp = test_sp.fillna(0)print('Train:', train_sp.count(), '| Test:', test_sp.count())# Kolom fitur (semua kecuali ID dan target)FEATURE_COLS = [c for c in train_sp.columns if c not in ['SUBJECT_ID', 'DEFAULT_FLAG']]print('Feature cols for training:', len(FEATURE_COLS))

## 5. Model Training: XGBoost (Snowpark ML Distributed)Gunakan `snowflake.ml.modeling.xgboost.XGBClassifier` — training berjalan **di warehouse** (distributed), bukan di client.

In [ ]:
from snowflake.ml.modeling.xgboost import XGBClassifierfrom snowflake.ml.modeling.metrics import roc_auc_score as snow_aucxgb_model = XGBClassifier(    input_cols=FEATURE_COLS,    label_cols=[TARGET],    output_cols=['XGB_PREDICTION'],    n_estimators=300,    max_depth=6,    learning_rate=0.1,    subsample=0.8,    colsample_bytree=0.8,    scale_pos_weight=10,  # ~10:1 class imbalance    random_state=42)print('Training XGBoost (distributed di warehouse)...')xgb_model.fit(train_sp)print('XGBoost trained!')

In [ ]:
# Evaluate XGBoostxgb_preds = xgb_model.predict(test_sp)xgb_preds_pd = xgb_preds.select('DEFAULT_FLAG', 'XGB_PREDICTION').to_pandas()from sklearn.metrics import roc_auc_score, classification_reportxgb_auc = roc_auc_score(xgb_preds_pd['DEFAULT_FLAG'], xgb_preds_pd['XGB_PREDICTION'])print(f'XGBoost - AUC: {xgb_auc:.4f} | Gini: {2*xgb_auc-1:.4f}')print(classification_report(xgb_preds_pd['DEFAULT_FLAG'], xgb_preds_pd['XGB_PREDICTION'], digits=3))

## 6. Model Training: LightGBM (Snowpark ML Distributed)

In [ ]:
from snowflake.ml.modeling.lightgbm import LGBMClassifierlgbm_model = LGBMClassifier(    input_cols=FEATURE_COLS,    label_cols=[TARGET],    output_cols=['LGBM_PREDICTION'],    n_estimators=400,    num_leaves=48,    learning_rate=0.05,    subsample=0.8,    colsample_bytree=0.8,    class_weight='balanced',    random_state=42,    verbose=-1)print('Training LightGBM (distributed di warehouse)...')lgbm_model.fit(train_sp)print('LightGBM trained!')

In [ ]:
lgbm_preds = lgbm_model.predict(test_sp)lgbm_preds_pd = lgbm_preds.select('DEFAULT_FLAG', 'LGBM_PREDICTION').to_pandas()lgbm_auc = roc_auc_score(lgbm_preds_pd['DEFAULT_FLAG'], lgbm_preds_pd['LGBM_PREDICTION'])print(f'LightGBM - AUC: {lgbm_auc:.4f} | Gini: {2*lgbm_auc-1:.4f}')

## 7. Model Training: Logistic Regression (Stepwise)Stepwise forward selection belum tersedia di Snowpark ML.Kita gunakan pandas sampling + sklearn LR untuk step ini (realistic approach).

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import StandardScaler as SkScaler# Tarik subset ke pandas untuk stepwiseLR_SAMPLE = 100_000train_lr_pd = train_sp.sample(n=LR_SAMPLE).to_pandas()test_lr_pd = test_sp.to_pandas()NUM_ONLY = [c for c in FEATURE_COLS if not any(cat in c for cat in CAT_COLS)]# Univariate AUC rankinguni = {}for c in NUM_ONLY[:50]:    try:        uni[c] = abs(roc_auc_score(train_lr_pd[TARGET], train_lr_pd[c].fillna(0)) - 0.5)    except:        uni[c] = 0ranked = sorted(uni, key=uni.get, reverse=True)[:30]print('Top 10 features:', ranked[:10])

In [ ]:
from sklearn.model_selection import train_test_split as ttsXa, Xv, ya, yv = tts(train_lr_pd[ranked], train_lr_pd[TARGET], test_size=0.3, random_state=7, stratify=train_lr_pd[TARGET])scaler = SkScaler()Xa_s = scaler.fit_transform(Xa.fillna(0))Xv_s = scaler.transform(Xv.fillna(0))# Forward stepwiseselected_idx, best_auc = [], 0.5for _ in range(20):    trial_best, trial_col = best_auc, None    for i, c in enumerate(ranked):        if i in selected_idx: continue        cols = selected_idx + [i]        lr = LogisticRegression(max_iter=200, C=1.0)        lr.fit(Xa_s[:, cols], ya)        auc = roc_auc_score(yv, lr.predict_proba(Xv_s[:, cols])[:, 1])        if auc > trial_best + 1e-4:            trial_best, trial_col = auc, i    if trial_col is None: break    selected_idx.append(trial_col); best_auc = trial_best    print(f'+ {ranked[trial_col]:30s} -> AUC {best_auc:.4f} ({len(selected_idx)} feats)')selected_features = [ranked[i] for i in selected_idx]print('\nSelected features:', selected_features)

In [ ]:
# Final LR on full trainX_train_lr = scaler.fit_transform(train_lr_pd[selected_features].fillna(0))X_test_lr = scaler.transform(test_lr_pd[selected_features].fillna(0))lr_final = LogisticRegression(max_iter=500, C=1.0)lr_final.fit(X_train_lr, train_lr_pd[TARGET])lr_pred = lr_final.predict_proba(X_test_lr)[:, 1]lr_auc = roc_auc_score(test_lr_pd[TARGET], lr_pred)print(f'Logistic Regression (stepwise) - AUC: {lr_auc:.4f} | Gini: {2*lr_auc-1:.4f}')

## 8. Model Comparison & ROC Curves

In [ ]:
from sklearn.metrics import roc_curveresults = pd.DataFrame({    'Model': ['LogReg (stepwise)', 'XGBoost (Snowpark ML)', 'LightGBM (Snowpark ML)'],    'AUC':  [lr_auc, xgb_auc, lgbm_auc],    'Gini': [2*lr_auc-1, 2*xgb_auc-1, 2*lgbm_auc-1],}).sort_values('AUC', ascending=False).reset_index(drop=True)display(results)plt.figure(figsize=(7,5))# LRfpr, tpr, _ = roc_curve(test_lr_pd[TARGET], lr_pred)plt.plot(fpr, tpr, label=f'LogReg (AUC={lr_auc:.3f})')# XGBfpr, tpr, _ = roc_curve(xgb_preds_pd['DEFAULT_FLAG'], xgb_preds_pd['XGB_PREDICTION'])plt.plot(fpr, tpr, label=f'XGBoost (AUC={xgb_auc:.3f})')# LGBMfpr, tpr, _ = roc_curve(lgbm_preds_pd['DEFAULT_FLAG'], lgbm_preds_pd['LGBM_PREDICTION'])plt.plot(fpr, tpr, label=f'LightGBM (AUC={lgbm_auc:.3f})')plt.plot([0,1],[0,1],'k--',alpha=0.4)plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curves - PD Models (Snowpark ML)')plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 9. Register Best Model ke Model RegistryGunakan `enable_monitoring=True` untuk menyiapkan ML Observability.

In [ ]:
from snowflake.ml.registry import Registrybest_name = results.iloc[0]['Model']best_model = xgb_model if 'XGBoost' in best_name else lgbm_model# Registry dengan monitoring enabledreg = Registry(    session=session,    database_name=DB,    schema_name=SCHEMA,    options={'enable_monitoring': True})# Log model terbaikmv = reg.log_model(    model=best_model,    model_name='CLIK_PD_MODEL',    version_name='V2_SNOWPARK_ML',    sample_input_data=train_sp.drop(['SUBJECT_ID', 'DEFAULT_FLAG']).limit(100),    comment=f'PD model ({best_name}) trained with Snowpark ML. AUC={results.iloc[0]["AUC"]:.4f}',    conda_dependencies=['xgboost', 'lightgbm', 'scikit-learn'],)# Set metricsmv.set_metric(metric_name='Test_AUC', value=float(results.iloc[0]['AUC']))mv.set_metric(metric_name='Test_Gini', value=float(results.iloc[0]['Gini']))print('Registered:', mv.model_name, mv.version_name)# Set as default versionm = reg.get_model('CLIK_PD_MODEL')m.default = 'V2_SNOWPARK_ML'

## 10. ML Monitoring / MLOpsSnowflake Model Monitor tracks:- **Prediction drift** (distribusi prediksi bergeser dari baseline)- **Feature drift** (distribusi input berubah)- **Performance metrics** (accuracy/AUC jika ground truth tersedia)Ref: end-to-end-ml-workflow guide — `CREATE MODEL MONITOR`

In [ ]:
# Simpan train & test data sebagai tabel untuk monitoring baseline & source# Tambahkan kolom prediksi dan timestamp# 1) Baseline = training data + predictionstrain_with_preds = xgb_model.predict(train_sp).with_column('SCORED_AT', lit('2026-07-01 00:00:00').cast('TIMESTAMP_NTZ'))train_with_preds.write.save_as_table('PD_MONITOR_BASELINE', mode='overwrite')print('Baseline table saved:', session.table('PD_MONITOR_BASELINE').count(), 'rows')# 2) Source = test/production data + predictions (simulates production scoring)test_with_preds = xgb_model.predict(test_sp).with_column('SCORED_AT', lit('2026-07-15 00:00:00').cast('TIMESTAMP_NTZ'))test_with_preds.write.save_as_table('PD_MONITOR_SOURCE', mode='overwrite')print('Source table saved:', session.table('PD_MONITOR_SOURCE').count(), 'rows')

In [ ]:
# Buat Model Monitor via SQLsession.sql('''CREATE OR REPLACE MODEL MONITOR CLIK_PD_MODEL_MONITORWITH    MODEL = CLIK_PD_MODEL    VERSION = V2_SNOWPARK_ML    FUNCTION = predict    SOURCE = CLIK_WORKSHOP2.PUBLIC.PD_MONITOR_SOURCE    BASELINE = CLIK_WORKSHOP2.PUBLIC.PD_MONITOR_BASELINE    TIMESTAMP_COLUMN = SCORED_AT    PREDICTION_CLASS_COLUMNS = (XGB_PREDICTION)    ACTUAL_CLASS_COLUMNS = (DEFAULT_FLAG)    ID_COLUMNS = (SUBJECT_ID)    WAREHOUSE = GEN2_SMALL    REFRESH_INTERVAL = '1 hour'    AGGREGATION_WINDOW = '1 day'''').collect()print('Model Monitor created: CLIK_PD_MODEL_MONITOR')

In [ ]:
# Query drift metricsdrift_df = session.sql('''SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(    'CLIK_PD_MODEL_MONITOR',    'DIFFERENCE_OF_MEANS',    'XGB_PREDICTION',    '1 DAY',    DATEADD(DAY, -30, CURRENT_DATE()),    CURRENT_DATE()))''')drift_df.show()print('\nDrift analysis complete. Monitor juga bisa dilihat di Snowsight > AI & ML > Models.')

## 11. Test Inference dari Registry

In [ ]:
# Batch inference via model registrytest_sdf = session.table('SUBJECT_FEATURES').limit(10)pred_sdf = mv.run(test_sdf, function_name='predict')pred_sdf.select('SUBJECT_ID', '"output_feature_0"').show()

## Summary - Snowflake ML/MLOps Stack yang Digunakan| Layer | sklearn (sebelumnya) | Snowpark ML (sekarang) ||-------|---------------------|------------------------|| Preprocessing | `sklearn.preprocessing.OneHotEncoder` | `snowflake.ml.modeling.preprocessing.OneHotEncoder` || Train/Test Split | `sklearn.model_selection.train_test_split` | `DataFrame.random_split()` || Training | `xgboost.XGBClassifier` (local) | `snowflake.ml.modeling.xgboost.XGBClassifier` (distributed) || Feature Management | Manual column selection | **Feature Store** (Entity, FeatureView, generate_dataset) || Model Storage | File-based / pickle | **Model Registry** (versioning, metrics, tags) || Monitoring | Manual / none | **Model Monitor** (drift detection, performance tracking) || Deployment | Custom SPCS | **Model Serving** (create_service REST API) |Lanjut ke:- **04a** — Batch scoring (warehouse)- **04b** — Real-time inference (SPCS)